In [23]:
import json
import os
from glob import glob
from collections import defaultdict
import pandas as pd
import re

In [24]:
models = glob('results/*.json')
models_mean = [model for model in models if 'mean' in model]
models_last = [model for model in models if 'last' in model]
models = models_mean + models_last

In [25]:
models = [
    'results/all_mpnet_base_mean.json',
    'results/BAAIbge_large_en_v1.5_mean.json',
    'results/google_bertbert_base_uncased_mean.json',
    'results/Alibaba_NLPgte_Qwen2_7B_instruct_lasttoken.json',
    'results/intfloate5_mistral_7b_instruct_mean.json',
    
    # 'results/BAAIbge_large_en_v1.5_2025_01_09_20_50.json',
    # 'results/google_bertbert_base_uncased_2025_01_09_19_54.json', 
    # 'results/all_mpnet_base_v2_2025_01_09_18_09.json'
]

In [26]:
all_models = {}
for model in models:
    all_scores = defaultdict(list)
    with open(model, 'r') as f:
        data = f.read().split('\n')
        scores = json.loads(data[0])
        params = json.loads(''.join(data[1:]))
        scores['model_id'] = model.replace('results/', '')
        scores['batch_size'] = params['per_device_train_batch_size']
        for key in scores:
            all_scores[key].append(scores[key]) 
    all_models[model] = pd.DataFrame(all_scores)

In [29]:
cols = all_models[list(all_models.keys())[0]].columns
keywords = [f'cosine_{metric}_before' for metric in ['accuracy@1', 'map@100', 'ndcg@10']]
sel_cols_before = [col for col in cols for keyword in keywords if keyword in col]
keywords = [f'cosine_{metric}_after' for metric in ['accuracy@1', 'map@100', 'ndcg@10']]
sel_cols_after = [col for col in cols for keyword in keywords if keyword in col]

In [30]:
prev_task_name = ''
for col_before, col_after in zip(sel_cols_before, sel_cols_after):
    split = col_before.split('_test_')
    task_name = split[0]
    col_before_format = col_before.replace(task_name + '_test_cosine_', '')
    col_after_format = col_after.replace(task_name + '_test_cosine_', '')
    if prev_task_name != task_name:
        print(task_name)
    prev_task_name = task_name
    metric = '_'.join(col_before.split('_')[-3:-1])
    print(f'\t{metric}')
    for model in all_models:
        df = all_models[model]
        # print(f'\t{model}')
        prev_task_name = task_name
        model = model.replace('results/', '').replace('.json', '')
        model = re.sub('_[0-9]*_[0-9]*_[0-9]*_[0-9]*_[0-9]*', '', model)
        print(f'\t\t{model}, before: {df.iloc[0][col_before]}, after: {df.iloc[0][col_after]}')

eq_to_category
	cosine_accuracy@1
		all_mpnet_base_mean, before: 0.0, after: 100.0
		BAAIbge_large_en_v1.5_mean, before: 22.22, after: 88.89
		google_bertbert_base_uncased_mean, before: 0.0, after: 77.78
		Alibaba_NLPgte_Qwen2_7B_instruct_lasttoken, before: 0.0, after: 88.89
		intfloate5_mistral_7b_instruct_mean, before: 44.44, after: 77.78
	cosine_ndcg@10
		all_mpnet_base_mean, before: 6.99, after: 88.95
		BAAIbge_large_en_v1.5_mean, before: 9.8, after: 90.9
		google_bertbert_base_uncased_mean, before: 0.0, after: 86.78
		Alibaba_NLPgte_Qwen2_7B_instruct_lasttoken, before: 3.56, after: 74.38
		intfloate5_mistral_7b_instruct_mean, before: 54.7, after: 85.99
	cosine_map@100
		all_mpnet_base_mean, before: 2.82, after: 81.83
		BAAIbge_large_en_v1.5_mean, before: 4.59, after: 83.46
		google_bertbert_base_uncased_mean, before: 0.0, after: 76.17
		Alibaba_NLPgte_Qwen2_7B_instruct_lasttoken, before: 5.37, after: 68.82
		intfloate5_mistral_7b_instruct_mean, before: 41.87, after: 78.91
eq_to_cl